# Лабораторная работа № 2. Перераспределение пустых контейнеров

Вариант 11 · Google Colab · **Среда выполнения → Выполнить всё**.

In [ ]:
%pip install -q numpy==2.3.5 scipy==1.17.0 matplotlib==3.10.8

## Исходные данные

Излишки, дефициты и дуги заданы шаблоном. Стоимости и ёмкости дуг получены небольшим случайным разбросом. Используем отдельный `data.json`. Если файла нет, создаём его с `seed=10012`.

In [ ]:
import csv
import json
import os
from contextlib import redirect_stderr, redirect_stdout
from ctypes import CDLL
from pathlib import Path
from heapq import heappop, heappush
from io import BytesIO

import numpy as np
from IPython.display import JSON, Markdown, display
from IPython.display import Image
from scipy.optimize import linprog

SEED = 10012
TOL = 1e-7
STATUS = {0: "optimal", 1: "limit", 2: "infeasible", 3: "unbounded", 4: "solver_error"}

LAB_DIR = Path.cwd()
if LAB_DIR.name != "lr2":
    LAB_DIR = LAB_DIR / "lr2"
DATA_PATH = LAB_DIR / "data.json"
RESULTS = LAB_DIR / "results"
STAR_RESULTS = RESULTS / "star"


def ports(rng):
    # N1–N3 имеют излишки, N4–N6 — нехватку контейнеров.
    sources = np.array([40, 35, 25])
    demand = np.array([30, 35, 35])
    base_arcs = [(1, 4, 25, 19), (1, 5, 35, 40), (1, 6, 40, 24),
                 (2, 4, 35, 24), (2, 5, 40, 26), (2, 6, 40, 55),
                 (3, 4, 35, 47), (3, 5, 15, 20), (3, 6, 35, 30),
                 (1, 2, 15, 13), (4, 6, 18, 14), (5, 6, 17, 25)]
    arcs = [
        {"tail": i, "head": j, "capacity": cap + int(rng.integers(-2, 3)),
         "cost": cost + int(rng.integers(0, 3))}
        for i, j, cap, cost in base_arcs
    ]
    return {"seed": SEED, "balance": np.r_[sources, -demand].tolist(), "arcs": arcs}


if not DATA_PATH.is_file():
    LAB_DIR.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(np.random.SeedSequence([SEED, 2]))
    DATA_PATH.write_text(
        json.dumps(ports(rng), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
data = json.loads(DATA_PATH.read_text(encoding="utf-8"))

## Решатель и сохранение результатов

In [ ]:
def write_csv(folder, name, headers, rows):
    path = folder / f"{name}.csv"
    path.parent.mkdir(exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(headers)
        writer.writerows(rows)


def write_check(folder, name, result):
    path = folder / f"{name}.json"
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def solve_lp(cost, *, label, verbose=True, **kwargs):
    print(f"\n=== {label} ===", flush=True)
    result = linprog(
        cost, method="highs-ds", options={"presolve": True, "disp": verbose}, **kwargs
    )
    print(f"{STATUS.get(result.status, result.status)} | {result.message}", flush=True)
    return result


def require_status(result, *allowed):
    if result.status not in allowed:
        raise RuntimeError(result.message)


def run_with_log(run, data, folder):
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / "solver.log").open("w", encoding="utf-8", buffering=1) as log:
        # HiGHS пишет лог напрямую в stdout, минуя Python.
        stdout_fd = os.dup(1)
        try:
            os.dup2(log.fileno(), 1)
            with redirect_stdout(log), redirect_stderr(log):
                return run(data)
        finally:
            CDLL(None).fflush(None)
            os.dup2(stdout_fd, 1)
            os.close(stdout_fd)


def show_table(path):
    with path.open(encoding="utf-8", newline="") as file:
        header, *rows = csv.reader(file)
    table = [header, ["---"] * len(header), *rows]
    display(Markdown("\n".join("| " + " | ".join(row) + " |" for row in table)))

## Построение модели

In [ ]:
def inputs(data):
    ports = np.array([(arc["tail"], arc["head"]) for arc in data["arcs"]]) - 1
    return (ports, np.array([arc["cost"] for arc in data["arcs"]]),
            np.array([arc["capacity"] for arc in data["arcs"]]),
            np.array(data["balance"]))


def incidence(ports, count):
    matrix = np.zeros((count, len(ports)))
    columns = np.arange(len(ports))
    matrix[ports[:, 0], columns] = 1
    matrix[ports[:, 1], columns] = -1
    return matrix


def solve(ports, cost, capacity, balance, label):
    matrix = incidence(ports, len(balance))
    # Последняя строка баланса следует из остальных и суммы b = 0.
    if abs(balance.sum()) > TOL:
        raise ValueError("Port balances must sum to zero")
    return solve_lp(cost, label=label, A_eq=matrix[:-1], b_eq=balance[:-1],
                    bounds=np.column_stack((np.zeros(len(ports)), capacity)))

## Независимая проверка и дополнительные расчёты

In [ ]:
def max_violation(values):
    return float(np.max(values, initial=0))


def verify_objective(checks, expected, actual):
    checks["objective_recomputed"] = float(actual)
    checks["objective_error"] = float(abs(expected - actual))
    if any(value > TOL for key, value in checks.items() if key != "objective_recomputed"):
        raise ValueError(f"Independent check failed: {checks}")
    return checks


def verify(flow, ports, cost, capacity, balance, objective):
    net = np.zeros(len(balance))
    np.add.at(net, ports[:, 0], flow)
    np.add.at(net, ports[:, 1], -flow)
    checks = {
        "port_balance": max_violation(abs(net - balance)),
        "arc_upper": max_violation(flow - capacity),
        "arc_lower": max_violation(-flow),
    }
    return verify_objective(checks, objective, np.dot(cost, flow)), net


def shortest_delivery(ports, cost, spare, need):
    queue = [(0, int(source), (), int(source)) for source in np.flatnonzero(need > TOL)]
    best = set()
    while queue:
        tariff, source, path, node = heappop(queue)
        if node in best:
            continue
        best.add(node)
        if need[node] < -TOL:
            return source, node, path
        for index in np.flatnonzero((ports[:, 0] == node) & (spare > TOL)):
            head = int(ports[index, 1])
            heappush(queue, (tariff + cost[index], source, (*path, index), head))
    raise ValueError("Greedy routing cannot satisfy remaining demand")


def greedy(ports, cost, capacity, balance):
    spare, need = capacity.astype(float), balance.astype(float)
    flow = np.zeros(len(ports))
    while np.max(need) > TOL:
        source, target, path = shortest_delivery(ports, cost, spare, need)
        edges = list(path)
        amount = min(need[source], -need[target], *spare[edges])
        need[source] -= amount
        need[target] += amount
        flow[edges] += amount
        spare[edges] -= amount
    return flow


def draw_network(ports, capacity, flow, balance):
    import matplotlib.pyplot as plt

    plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10})
    pos = np.array([(0, 3), (0, 1.5), (0, 0), (4.8, 3), (4.8, 1.5), (4.8, 0)])
    labels = [(1.15, 3.18), (1.32, 2.57), (2.78, 1.26), (1.35, 1.88),
              (1.98, 1.50), (1.1, 0.99), (2.8, 1.89), (3.9, 1.18), (2.8, -0.14),
              (-0.27, 2.23), (5.84, 1.56), (5.03, 0.57)]
    fig, ax = plt.subplots(figsize=(6.4, 3.28))
    for e, ((i, j), (lx, ly)) in enumerate(zip(ports, labels)):
        active = flow[e] > TOL
        bend = 0.13 if e == 9 else (-0.55 if e == 10 else 0)
        ax.annotate("", xy=pos[j], xytext=pos[i],
                    arrowprops=dict(arrowstyle="-|>", color="#171717" if active else "#777777",
                                    linestyle="-" if active else "--",
                                    linewidth=1.45 if active else 0.8, shrinkA=14, shrinkB=14,
                                    connectionstyle=f"arc3,rad={bend}"), zorder=1)
        ax.text(lx, ly, f"{flow[e]:g}/{capacity[e]:g}", ha="center", va="center", zorder=3,
                bbox=dict(boxstyle="round,pad=0.14", fc="white", ec="none"))
    for k, (x, y) in enumerate(pos):
        ax.scatter(x, y, s=700, facecolor="white", edgecolor="#171717", linewidth=1.2, zorder=4)
        ax.text(x, y, f"П{k + 1}", ha="center", va="center", zorder=5)
        ax.text(x - 0.29 if k < 3 else x + 0.31, y + 0.26, f"{balance[k]:+g}",
                ha="right" if k < 3 else "left", zorder=5)
    ax.set(xlim=(-0.85, 6.35), ylim=(-0.39, 3.49))
    ax.axis("off")
    fig.savefig(RESULTS / "network.pdf", bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)
    return fig

## Эксперименты

Базовый план, перегрузка и отказ самой загруженной дуги, рост спроса и сравнение с жадным планом.

In [ ]:
def run_lab(data):
    ports, cost, capacity, balance = inputs(data)
    base = solve(ports, cost, capacity, balance, "base")
    require_status(base, 0)
    # При одинаковой загрузке в процентах выбираем больший поток.
    busiest = max(range(len(ports)), key=lambda i: (base.x[i] / capacity[i], base.x[i], -i))
    congested, failed = capacity.astype(float), capacity.copy()
    congested[busiest] *= 0.75
    failed[busiest] = 0
    growth = balance * 1.2
    summary, solution, arc_rows, node_rows, checks = [], [], [], [], {}
    base_cost = base.fun
    for name, cap, b, result in [
        ("base", capacity, balance, base),
        ("congestion", congested, balance, None),
        ("growth", capacity, growth, None),
        ("failure", failed, balance, None),
    ]:
        if result is None:
            result = solve(ports, cost, cap, b, name)
        require_status(result, 0)
        summary.append([name, cap[busiest], -b[b < 0].sum(), STATUS[result.status], result.fun,
                        result.fun - base_cost, result.nit])
        checks[name], net = verify(result.x, ports, cost, cap, b, result.fun)
        potentials = np.r_[result.eqlin.marginals, 0]
        for k, b_k in enumerate(b):
            node_rows.append([name, k + 1, b_k, net[k], net[k] - b_k, potentials[k]])
        for e, (i, j) in enumerate(ports):
            solution.append([name, i + 1, j + 1, cap[e], result.x[e], cost[e], cost[e] * result.x[e]])
            if name == "base":
                arc_rows.append([i + 1, j + 1, cap[e] - result.x[e], result.upper.marginals[e]])
    heuristic = greedy(ports, cost, capacity, balance)
    greedy_cost = cost @ heuristic
    checks["greedy"], _ = verify(heuristic, ports, cost, capacity, balance, greedy_cost)
    summary.append(["greedy", capacity[busiest], -balance[balance < 0].sum(), "feasible",
                    greedy_cost, greedy_cost - base_cost, ""])
    for e, (i, j) in enumerate(ports):
        solution.append(["greedy", i + 1, j + 1, capacity[e], heuristic[e], cost[e],
                         cost[e] * heuristic[e]])
    write_csv(RESULTS, "scenarios",
              ["scenario", "busiest_arc_capacity", "demand", "status", "cost", "delta", "iterations"],
              summary)
    write_csv(RESULTS, "solution", ["scenario", "tail", "head", "capacity", "flow", "tariff", "cost"],
              solution)
    write_csv(RESULTS, "nodes", ["scenario", "port", "balance", "net_flow", "error", "potential"],
              node_rows)
    write_csv(RESULTS, "arc_marginals", ["tail", "head", "slack", "marginal"], arc_rows)
    write_check(RESULTS, "checks", checks)
    return draw_network(ports, capacity, base.x, balance)


figure = run_with_log(run_lab, data, RESULTS)

## Результаты

Планы, проверки и лог сохранены в папке `results`.

In [ ]:
show_table(RESULTS / "scenarios.csv")
with BytesIO() as buffer:
    figure.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
display(JSON(json.loads((RESULTS / "checks.json").read_text(encoding="utf-8"))))

## ★ Двойственная модель

Решаем двойственную задачу и проверяем допустимость, дополняющую нежёсткость и равенство целей.

In [ ]:
def solve_dual(ports, cost, capacity, balance):
    count = len(balance)
    matrix = incidence(ports, count)
    bounds = [(None, None)] * (count - 1) + [(0, 0)] + [(0, None)] * len(cost)
    return solve_lp(np.r_[-balance, capacity], label="dual: maximize b*p - u*l",
                    A_ub=np.column_stack((matrix.T, -np.eye(len(cost)))),
                    b_ub=cost, bounds=bounds)


def run_star(data):
    ports, cost, capacity, balance = inputs(data)
    primal = solve(ports, cost, capacity, balance, "primal: base")
    dual = solve_dual(ports, cost, capacity, balance)
    require_status(primal, 0)
    require_status(dual, 0)
    pi, mu = np.split(dual.x, [len(balance)])
    tail, head = ports.T
    reduced_cost = cost - pi[tail] + pi[head] + mu
    objective = balance @ pi - capacity @ mu
    checks, _ = verify(primal.x, ports, cost, capacity, balance, primal.fun)
    checks.update({
        "dual_feasibility": max(max_violation(-reduced_cost), max_violation(-mu)),
        "arc_lower_complementarity": max_violation(abs(reduced_cost * primal.x)),
        "arc_upper_complementarity": max_violation(abs(mu * (capacity - primal.x))),
        "dual_objective_error": abs(objective + dual.fun),
        "duality_gap": abs(primal.fun - objective),
    })
    assert all(value <= TOL for key, value in checks.items() if key != "objective_recomputed")
    checks["dual_objective"] = float(objective)
    write_check(STAR_RESULTS, "checks", checks)
    write_csv(STAR_RESULTS, "potentials", ["port", "balance", "potential", "contribution"],
              ((i + 1, b, p, b * p) for i, (b, p) in enumerate(zip(balance, pi))))
    write_csv(STAR_RESULTS, "arc_multipliers", ["tail", "head", "capacity", "flow", "mu", "reduced_cost"],
              ((i + 1, j + 1, u, x, m, r)
               for (i, j), u, x, m, r in zip(ports, capacity, primal.x, mu, reduced_cost)))


run_with_log(run_star, data, STAR_RESULTS)
show_table(STAR_RESULTS / "potentials.csv")
display(JSON(json.loads((STAR_RESULTS / "checks.json").read_text(encoding="utf-8"))))